# Step 1: Fine-tuning BLIP for Fashion Product Image Captioning

## Overview
This notebook fine-tunes **`Salesforce/blip-image-captioning-base`** on the **`ashraq/fashion-product-images-small`** dataset.

The fine-tuned model learns to generate descriptive captions from fashion product images (e.g., "Navy Blue Men's Casual Shirt"). The generated caption is then passed to the Step 2 GPT-2 model to produce advertising copy.

### Pipeline
```
Product Image → [Fine-tuned BLIP] → Caption → [Fine-tuned GPT-2] → Ad Copy
```

### Dataset
- **Source**: `ashraq/fashion-product-images-small`
- **Size**: 44,072 fashion product images
- **Caption target**: `productDisplayName` field (e.g., "Turtle Check Men Navy Blue Shirt")

### Model
- **Base model**: `Salesforce/blip-image-captioning-base`
- **Task**: Image-to-text (conditional image captioning)
- **Output**: Pushed to HuggingFace Hub

In [ ]:
# ── Install dependencies (Colab / fresh environment) ───────────────────────
!pip install -q transformers datasets accelerate huggingface_hub Pillow torch torchvision
!pip install -q evaluate rouge_score

In [ ]:
import os
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import (
    BlipProcessor,
    BlipForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from datasets import load_dataset
import evaluate

# ── Device ──────────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── HuggingFace Hub login (required to push model) ────────────────────────
# Replace with your own token from https://huggingface.co/settings/tokens
from huggingface_hub import login

HF_TOKEN = "hf_YOUR_TOKEN_HERE"   # <── paste your HF write token
HF_USERNAME = "JescYip"           # <── your HuggingFace username
MODEL_REPO = f"{HF_USERNAME}/blip-fashion-finetuned"

login(token=HF_TOKEN)
print(f"Model will be pushed to: {MODEL_REPO}")

## 1. Load & Explore Dataset

In [ ]:
# ── Load the fashion product images dataset ──────────────────────────────
print("Loading ashraq/fashion-product-images-small ...")
raw_dataset = load_dataset("ashraq/fashion-product-images-small", split="train")
print(f"Total samples: {len(raw_dataset)}")
print(f"Features: {list(raw_dataset.features.keys())}")

In [ ]:
# ── Explore a few samples ──────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, ax in enumerate(axes):
    row = raw_dataset[i]
    ax.imshow(row['image'])
    ax.set_title(row['productDisplayName'], fontsize=8, wrap=True)
    ax.axis('off')
    print(f"[{i}] {row['gender']} | {row['articleType']} | {row['baseColour']} | {row['usage']}")
    print(f"     Caption target: {row['productDisplayName']}")
plt.tight_layout()
plt.show()

In [ ]:
# ── Build caption strings ─────────────────────────────────────────────────
# Strategy: use productDisplayName as the primary target caption.
# If empty, fall back to a structured sentence from attributes.

def build_caption(row):
    name = (row.get('productDisplayName') or '').strip()
    if name:
        return name
    # Fallback: structured sentence
    parts = [
        row.get('gender', ''),
        row.get('baseColour', ''),
        row.get('articleType', ''),
        f"for {row.get('usage', '')} use" if row.get('usage') else '',
    ]
    return ' '.join(p for p in parts if p).strip()

# Add caption column
raw_dataset = raw_dataset.map(
    lambda row: {"caption": build_caption(row)},
    desc="Building captions"
)

# Remove rows with empty captions
raw_dataset = raw_dataset.filter(lambda x: len(x["caption"]) > 3)
print(f"Dataset after cleaning: {len(raw_dataset)} samples")
print("\nSample captions:")
for i in range(5):
    print(f"  [{i}] {raw_dataset[i]['caption']}")

In [ ]:
# ── Train / Validation split ──────────────────────────────────────────────
# Use 5,000 samples for training speed (increase for better quality)
TRAIN_SIZE = 5000
VAL_SIZE   = 500

shuffled = raw_dataset.shuffle(seed=42)
train_data = shuffled.select(range(TRAIN_SIZE))
val_data   = shuffled.select(range(TRAIN_SIZE, TRAIN_SIZE + VAL_SIZE))

print(f"Train: {len(train_data)} | Val: {len(val_data)}")

## 2. Load BLIP Model & Processor

In [ ]:
# ── Load BLIP processor and model ─────────────────────────────────────────
BASE_MODEL = "Salesforce/blip-image-captioning-base"

print(f"Loading {BASE_MODEL} ...")
processor = BlipProcessor.from_pretrained(BASE_MODEL)
model = BlipForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters  : {total_params:,}")
print(f"Trainable params  : {trainable:,}")

## 3. PyTorch Dataset & Collator

In [ ]:
class FashionCaptionDataset(Dataset):
    """
    Wraps the HuggingFace fashion dataset and tokenises
    image + caption pairs for BLIP fine-tuning.
    """
    def __init__(self, hf_dataset, processor, max_caption_length=64):
        self.data      = hf_dataset
        self.processor = processor
        self.max_len   = max_caption_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row     = self.data[idx]
        image   = row["image"].convert("RGB")
        caption = row["caption"]

        # Encode image + caption together
        encoding = self.processor(
            images=image,
            text=caption,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        # Build labels: mask padding tokens with -100
        labels = encoding["input_ids"].squeeze().clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {
            "pixel_values": encoding["pixel_values"].squeeze(),
            "input_ids":    encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels":       labels,
        }


train_dataset = FashionCaptionDataset(train_data, processor)
val_dataset   = FashionCaptionDataset(val_data,   processor)
print(f"Train dataset: {len(train_dataset)} | Val dataset: {len(val_dataset)}")

# Quick sanity check
sample = train_dataset[0]
print("\nSample keys:", list(sample.keys()))
print("pixel_values shape:", sample["pixel_values"].shape)
print("input_ids shape:"   , sample["input_ids"].shape)

## 4. Fine-Tuning with Seq2SeqTrainer

In [ ]:
# ── Evaluation metric: ROUGE-L ────────────────────────────────────────────
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
    # Decode predictions
    decoded_preds = processor.batch_decode(
        np.where(preds != -100, preds, processor.tokenizer.pad_token_id),
        skip_special_tokens=True
    )
    # Decode labels
    labels = np.where(labels != -100, labels, processor.tokenizer.pad_token_id)
    decoded_labels = processor.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )
    return {k: round(v, 4) for k, v in result.items()}

In [ ]:
# ── Training Arguments ────────────────────────────────────────────────────
OUTPUT_DIR = "./blip-fashion-finetuned"

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=200,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=50,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    predict_with_generate=True,
    generation_max_length=64,
    fp16=(DEVICE == "cuda"),
    report_to="none",
    push_to_hub=True,
    hub_model_id=MODEL_REPO,
    hub_token=HF_TOKEN,
)

# ── Trainer ───────────────────────────────────────────────────────────────
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Starting fine-tuning...")
trainer.train()

## 5. Evaluation & Inference

In [ ]:
# ── Final evaluation on validation set ───────────────────────────────────
metrics = trainer.evaluate()
print("\nEvaluation Results:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

In [ ]:
# ── Qualitative inference on unseen samples ───────────────────────────────
model.eval()
test_samples = raw_dataset.select(range(TRAIN_SIZE + VAL_SIZE, TRAIN_SIZE + VAL_SIZE + 5))

print("Inference demos (unseen samples):")
print("-" * 60)

for i, row in enumerate(test_samples):
    image   = row["image"].convert("RGB")
    gt_cap  = row["caption"]

    inputs = processor(images=image, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=50,
            num_beams=4,
        )
    pred_cap = processor.decode(out[0], skip_special_tokens=True)

    print(f"[{i+1}] Ground truth : {gt_cap}")
    print(f"     Predicted   : {pred_cap}")
    print()

## 6. Save & Push to HuggingFace Hub

In [ ]:
# ── Save locally ──────────────────────────────────────────────────────────
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

# ── Push to Hub ───────────────────────────────────────────────────────────
model.push_to_hub(MODEL_REPO, token=HF_TOKEN)
processor.push_to_hub(MODEL_REPO, token=HF_TOKEN)
print(f"\nModel pushed to HuggingFace Hub: https://huggingface.co/{MODEL_REPO}")

In [ ]:
# ── Verify: load from Hub and run inference ───────────────────────────────
from transformers import pipeline as hf_pipeline

captioner = hf_pipeline(
    "image-to-text",
    model=MODEL_REPO,
    token=HF_TOKEN
)

test_image = raw_dataset[TRAIN_SIZE + VAL_SIZE]["image"].convert("RGB")
result = captioner(test_image)
print(f"Hub model inference: {result[0]['generated_text']}")
print("\n✅ BLIP fine-tuning complete!")
print(f"Model available at: https://huggingface.co/{MODEL_REPO}")
print("The generated caption is the input for Step 2 (GPT-2 ad generation).")

## Notes

| Parameter | Value | Notes |
|-----------|-------|-------|
| Base Model | `Salesforce/blip-image-captioning-base` | 224M params |
| Dataset | `ashraq/fashion-product-images-small` | 44,072 images |
| Training subset | 5,000 images | Increase for better quality |
| Epochs | 5 | Adjust based on GPU budget |
| Batch size | 8 | Per device |
| Learning rate | 5e-5 | With cosine decay |
| Precision | FP16 (GPU) | Reduces VRAM usage |

### How to use the trained model
```python
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image

processor = BlipProcessor.from_pretrained("JescYip/blip-fashion-finetuned")
model = BlipForConditionalGeneration.from_pretrained("JescYip/blip-fashion-finetuned")

image = Image.open("product.jpg").convert("RGB")
inputs = processor(images=image, return_tensors="pt")
out = model.generate(**inputs, max_new_tokens=50)
caption = processor.decode(out[0], skip_special_tokens=True)
print(caption)  # e.g. "Navy Blue Men Casual Shirts"
```